In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib
import gc

df = pd.read_parquet("../data/processed/cleaned_flows.parquet")

rare_classes = ['Heartbleed', 'Infiltration', 'Web Attack - Sql Injection']
df = df[~df['Label'].isin(rare_classes)]

print(df['Label'].value_counts())
print(f"\nTotal classes: {df['Label'].nunique()}")

Label
BENIGN                      2095051
DoS Hulk                     172846
DDoS                         128014
PortScan                      90694
DoS GoldenEye                 10286
FTP-Patator                    5931
DoS slowloris                  5385
DoS Slowhttptest               5228
SSH-Patator                    3219
Bot                            1948
Web Attack - Brute Force       1470
Web Attack - XSS                652
Name: count, dtype: int64

Total classes: 12


In [2]:
label_encoder = LabelEncoder()
y_multi = label_encoder.fit_transform(df['Label'])

# save the mapping so you can decode predictions back to class names later
class_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(class_mapping)

joblib.dump(label_encoder, "../data/processed/label_encoder_multiclass.joblib")

{'BENIGN': np.int64(0), 'Bot': np.int64(1), 'DDoS': np.int64(2), 'DoS GoldenEye': np.int64(3), 'DoS Hulk': np.int64(4), 'DoS Slowhttptest': np.int64(5), 'DoS slowloris': np.int64(6), 'FTP-Patator': np.int64(7), 'PortScan': np.int64(8), 'SSH-Patator': np.int64(9), 'Web Attack - Brute Force': np.int64(10), 'Web Attack - XSS': np.int64(11)}


['../data/processed/label_encoder_multiclass.joblib']

In [3]:
feature_cols = [c for c in df.columns if c not in ['Label', 'Label_binary']]
X = df[feature_cols]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_multi, test_size=0.30, stratify=y_multi, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (1764506, 78), Val: (378109, 78), Test: (378109, 78)


In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "../data/processed/scaler_multiclass.joblib")
print("Scaling complete.")

Scaling complete.


In [5]:
np.save("../data/processed/X_train_multiclass.npy", X_train_scaled)
np.save("../data/processed/X_val_multiclass.npy", X_val_scaled)
np.save("../data/processed/X_test_multiclass.npy", X_test_scaled)
np.save("../data/processed/y_train_multiclass.npy", y_train)
np.save("../data/processed/y_val_multiclass.npy", y_val)
np.save("../data/processed/y_test_multiclass.npy", y_test)

print("Saved all multiclass preprocessing artifacts.")

Saved all multiclass preprocessing artifacts.
